In [1]:
import json
import csv
import math
import time
from pathlib import Path
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [2]:
DOCS_DIR        = "../../data/parsed_json/"
CHROMA_BASE     = "../../data/vector_db"
GOLDEN_SET_PATH = "../../eval/golden_dataset/golden_dataset_v2.csv"
COLLECTION_NAME = "rfp_docs"
MAX_CHUNK_SIZE  = 1000
BATCH_SIZE      = 500

# 실험할 모델 목록
MODELS_TO_RUN = [
    "bge-m3",
    "ko-sroberta-multitask",
    "text-embedding-3-small",
    #"text-embedding-3-large",
    #"embed-multilingual-v3",
]

# API 키 (필요한 모델만 설정)
import os
os.environ["OPENAI_API_KEY"] = ""
#os.environ["COHERE_API_KEY"] = "..."

MODEL_COST = {
    "text-embedding-3-small": 0.02,
    #"text-embedding-3-large": 0.13,
    "bge-m3":                 0.0,
    "ko-sroberta-multitask":  0.0,
    #"embed-multilingual-v3":  0.10,
}

In [3]:
GS_TO_DOCID = {
    "GKL_그룹웨어":            "D093",
    "KUSF_체육":               "D011",
    "강릉어선안전":             "D024",
    "경기_사회서비스":          "D087",
    "고려대학교_차세대포털":    "D008",
    "광주과기원_RCMS":          "D073",
    "광주과학기술원_학사시스템": "D039",
    "구미_육상":               "D018",
    "국립중앙의료원_응급":      "D069",
    "국민연금공단_이러닝":      "D049",
    "국민연금_멀티턴1":         "D050",
    "국민연금_멀티턴2":         "D050",
    "국민연금_멀티턴3":         "D050",
    "국방_대용량":             "D010",
    "기초과학연구원_극저온":    "D051",
    "나노종합_팹":             "D099",
    "대검찰청_홈페이지":        "D053",
    "민속박물관_아카이브":      "D090",
    "벤처협회_시스템":          "D086",
    "보험개발원_실손":          "D083",
    "봉화군_재난":             "D005",
    "부산관광_ERP":            "D037",
    "서민금융_채팅":            "D056",
    "서영대_교육":             "D045",
    "서울_디지털성범죄":        "D068",
    "서울_지도플랫폼":          "D040",
    "서울교육청_ISP":           "D084",
    "세종_인사":               "D088",
    "우즈벡_관개":             "D072",
    "울산_버스":               "D034",
    "인천_도시계획":            "D004",
    "인천_일자리":             "D030",
    "인천공항_ERP":            "D079",
    "적십자_재해복구":          "D095",
    "철도_ISP":               "D070",
    "통합정보시스템_충돌":      "D016",
    "평택_버스":               "D060",
    "해양박물관_자료":          "D066",
    # 다중 문서
    "고려대_vs_광주과기원":     ["D008", "D039"],
    "버스_다중비교":            ["D034", "D060"],
    "재난안전_종합":            ["D005", "D007"],
    "철도_vs_인천_ISP":        ["D070", "D030"],
    # 평가 제외
    "TEST": None, "unknown": None, "ISP_다중비교": None,
    "교육관련_다중문서": None, "문화_다중비교": None,
    "의료_다중문서": None, "재공고_종합": None,
    "신규_vs_고도화": None, "예산_최소_최대": None,
    "멀티턴_심화1": None, "멀티턴_심화2": None,
    "모른다_테스트1": None, "모른다_테스트2": None,
    "모른다_테스트3": None, "모른다_테스트4": None,
    "모른다_테스트5": None, "모른다_테스트6": None,
    "존재하지않는사업": None, "입찰마감_확인": None,
}

1. 유틸리티

In [4]:
def clean(val):
    # NaN / None → 빈 문자열 (Chroma 메타데이터는 NaN·None 불가)
    if val is None:
        return ""
    if isinstance(val, float) and math.isnan(val):
        return ""
    return val

In [5]:
def build_payload(doc: dict, section: dict, block: dict) -> dict:
    # 청크 메타데이터(페이로드) 생성
    meta = doc.get("metadata", {})
    return {
        "doc_id":        str(clean(doc.get("doc_id"))),
        "file_name":     str(clean(doc.get("file_name"))),
        "source_format": str(clean(doc.get("source_format"))),
        "사업명":         str(clean(meta.get("사업명"))),
        "발주기관":       str(clean(meta.get("발주기관"))),
        "사업유형":       str(clean(meta.get("사업유형"))),
        "사업금액":       float(clean(meta.get("사업금액")) or 0.0),
        "공고번호":       str(clean(meta.get("공고번호"))),
        "공고차수":       float(clean(meta.get("공고차수")) or 0.0),
        "공개일자":       str(clean(meta.get("공개일자"))),
        "입찰참여시작일":  str(clean(meta.get("입찰참여시작일"))),
        "입찰참여마감일":  str(clean(meta.get("입찰참여마감일"))),
        "재공고여부":     bool(meta.get("재공고여부", False)),
        "linked_doc_id": str(clean(meta.get("linked_doc_id"))),
        "사업요약":       str(clean(meta.get("사업요약"))),
        "header_path":   " > ".join(section.get("header_path", [])),
        "section_id":    str(clean(section.get("section_id"))),
        "block_id":      str(clean(block.get("block_id"))),
        "block_type":    str(clean(block.get("type"))),
        "table_type":    str(clean(block.get("table_type"))),
    }

2. 청킹

In [6]:
def chunk_section(section: dict) -> list[dict]:

    header_prefix = " > ".join(section.get("header_path", []))
    results = []
    current_text = ""
    last_text_block = None

    for block in section.get("blocks", []):
        if block.get("is_decorative"):
            continue
        if block["type"] == "table":
            # 쌓인 텍스트 먼저 방출
            if current_text.strip():
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                    "block":   last_text_block,
                })
                current_text = ""
                last_text_block = None
            # 표 단독 방출
            results.append({
                "content": f"[섹션: {header_prefix}]\n\n{block['content']}",
                "block":   block,
            })
        else:
            # 텍스트 블록 누적
            if len(current_text) + len(block["content"]) > MAX_CHUNK_SIZE and current_text.strip():
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                    "block":   last_text_block,
                })
                current_text = block["content"] + "\n\n"
            else:
                current_text += block["content"] + "\n\n"
            last_text_block = block

    # 남은 텍스트 방출
    if current_text.strip():
        results.append({
            "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
            "block":   last_text_block,
        })

    return results

In [7]:
def process_doc(doc: dict) -> tuple[list[str], list[dict]]:
    # JSON 문서 1개 → (청크 텍스트 리스트, 메타데이터 리스트)
    texts, metas = [], []

    warnings = doc.get("qa", {}).get("extraction_warnings", [])
    if warnings:
        print(f"  [WARN] {doc['doc_id']} — extraction_warnings: {warnings}")

    meta = doc.get("metadata", {})
    summary  = str(clean(meta.get("사업요약")))
    사업명   = str(clean(meta.get("사업명")))
    발주기관 = str(clean(meta.get("발주기관")))

    for section in doc.get("sections", []):
        chunks = chunk_section(section)
        for item in chunks:
            prefix = (
                f"[사업명] {사업명}\n"
                f"[발주기관] {발주기관}\n"
                f"[요약] {summary}\n\n"
            )
            texts.append(prefix + item["content"])
            metas.append(build_payload(doc, section, item["block"] or {}))
    return texts, metas

3. 임베딩 모델

In [8]:
def load_embedding_model(name: str):
    if name == "text-embedding-3-small":
        return OpenAIEmbeddings(model="text-embedding-3-small")
    elif name == "text-embedding-3-large":
        return OpenAIEmbeddings(model="text-embedding-3-large")
    elif name == "bge-m3":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "ko-sroberta-multitask":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="snunlp/KR-SBERT-V40K-klueNLI-augSTS",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "embed-multilingual-v3":
        from langchain_cohere import CohereEmbeddings
        return CohereEmbeddings(model="embed-multilingual-v3.0")
    else:
        raise ValueError(f"알 수 없는 임베딩 모델: {name}")

4. Vector DB / Retrieval

In [23]:
def embed_with_retry(vectorstore, texts, metadatas, max_retries=10):
    wait = 10
    for attempt in range(max_retries):
        try:
            vectorstore.add_texts(texts=texts, metadatas=metadatas)
            return
        except Exception as e:
            if "RateLimitError" in type(e).__name__ or "rate_limit" in str(e).lower():
                print(f"  [RateLimit] {wait}초 대기 후 재시도 ({attempt+1}/{max_retries})")
                time.sleep(wait)
                wait = min(wait * 2, 120)  # 10 → 20 → 40 → 80 → 120초
            else:
                raise
    raise RuntimeError("최대 재시도 횟수 초과")


def save_vectorstore_for_model(model_name: str, all_texts: list, all_metas: list):
    safe_name = model_name.replace("/", "_")
    chroma_dir = f"{CHROMA_BASE}__{safe_name}"

    if Path(chroma_dir).exists():
        embedding_model = load_embedding_model(model_name)
        existing_vs = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embedding_model,
            persist_directory=chroma_dir,
        )
        existing_count = existing_vs._collection.count()
        if existing_count >= len(all_texts):
            print(f"[SKIP] {model_name} — 완료된 DB 존재 ({existing_count}개 청크)")
            return
        else:
            print(f"[RESUME] {model_name} — {existing_count}/{len(all_texts)}개, 이어서 진행")
            start_from = existing_count
            vectorstore = existing_vs
    else:
        embedding_model = load_embedding_model(model_name)
        start_from = 0
        vectorstore = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embedding_model,
            persist_directory=chroma_dir,
        )

    for start in range(start_from, len(all_texts), BATCH_SIZE):
        end = start + BATCH_SIZE
        embed_with_retry(vectorstore, all_texts[start:end], all_metas[start:end])
        print(f"  저장 완료: {min(end, len(all_texts))}/{len(all_texts)}")

    print(f"  완료 ({vectorstore._collection.count()}개 청크) → {chroma_dir}")

5. 실행

In [24]:
docs_dir = Path(DOCS_DIR)
json_files = sorted(docs_dir.glob("*.json"))
print(f"JSON 파일 수: {len(json_files)}")

all_texts, all_metas = [], []
for json_file in json_files:
    with open(json_file, encoding="utf-8") as f:
        doc = json.load(f)
    texts, metas = process_doc(doc)
    all_texts.extend(texts)
    all_metas.extend(metas)

print(f"총 청크 수: {len(all_texts)}")

# 모델별 벡터스토어 저장
for model in MODELS_TO_RUN:
    save_vectorstore_for_model(model, all_texts, all_metas)

JSON 파일 수: 97
  [WARN] D002 — extraction_warnings: ['빈 표 감지 (blk#1)', 'high_decorative_table_ratio: 23.7%']
  [WARN] D003 — extraction_warnings: ['빈 표 감지 (blk#50)', '빈 표 감지 (blk#300)', 'high_decorative_table_ratio: 19.7%']
  [WARN] D004 — extraction_warnings: ['high_decorative_table_ratio: 16.5%']
  [WARN] D005 — extraction_warnings: ['빈 표 감지 (blk#1)', '빈 표 감지 (blk#101)', '빈 표 감지 (blk#315)', 'high_decorative_table_ratio: 26.4%']
  [WARN] D006 — extraction_warnings: ['high_decorative_table_ratio: 20.3%']
  [WARN] D007 — extraction_warnings: ['빈 표 감지 (blk#277)', '빈 표 감지 (blk#312)', '빈 표 감지 (blk#362)', '빈 표 감지 (blk#367)', '빈 표 감지 (blk#368)']
  [WARN] D009 — extraction_warnings: ['빈 표 감지 (blk#229)', '빈 표 감지 (blk#229)', '빈 표 감지 (blk#229)', 'high_decorative_table_ratio: 28.7%']
  [WARN] D010 — extraction_warnings: ['빈 표 감지 (blk#6)', 'high_decorative_table_ratio: 47.8%']
  [WARN] D011 — extraction_warnings: ['빈 표 감지 (blk#99)']
  [WARN] D012 — extraction_warnings: ['빈 표 감지 (blk#30)', '빈 표 감지 (

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[SKIP] bge-m3 — 완료된 DB 존재 (17382개 청크)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[SKIP] ko-sroberta-multitask — 완료된 DB 존재 (17382개 청크)
[SKIP] text-embedding-3-small — 완료된 DB 존재 (17382개 청크)


6. 검색 테스트

In [26]:
TEST_QUERIES = [
    "보안 요구사항이 있는 사업은?",
    "사업 예산이 가장 큰 사업은?",
    "클라우드 전환 관련 사업은?",
]
for VISUAL_MODEL in MODELS_TO_RUN:
    safe_name = VISUAL_MODEL.replace("/", "_")
    chroma_dir = f"{CHROMA_BASE}__{safe_name}"

    if not Path(chroma_dir).exists():
        print(f"[SKIP] {VISUAL_MODEL} — DB 없음")
        continue

    vs = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=load_embedding_model(VISUAL_MODEL),
        persist_directory=chroma_dir,
    )

    print(f"\n{'='*50}")
    print(f"[모델] {VISUAL_MODEL}")
    for q in TEST_QUERIES:
        print(f"\n[질의] {q}")
        results = vs.similarity_search_with_relevance_scores(q, k=TOP_K)
        for i, (doc, score) in enumerate(results, 1):
            m = doc.metadata
            print(f"  [{i}] score={score:.4f} | {m.get('사업명')} | {m.get('header_path')}")
            print(f"       {doc.page_content[:200]}...")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


[모델] bge-m3

[질의] 보안 요구사항이 있는 사업은?
  [1] score=0.5167 | 경영정보시스템 기능개선 | (서두) > 2. 요구사항 세부내용
       [사업명] 경영정보시스템 기능개선
[발주기관] 부산관광공사
[요약] - 사업개요: 부산관광공사 경영정보시스템 기능개선 용역
- 추진배경: 업무환경 변화에 따른 기능개선 필요
- 사업범위: 경영정보시스템 기능 개선 및 최적화
- 기대효과: 업무처리 효율성 및 편의성 향상
- 추진목표: 사용자 요구사항 반영 및 업무환경 개선

[섹션: (서두) > 2. 요구사...
  [2] score=0.4921 | [재공고]차세대 통합정보시스템(ERP) 구축 | Ⅲ. 일반사항 > 1. 상세 요구사항 > 11) 보안 요구사항
       [사업명] [재공고]차세대 통합정보시스템(ERP) 구축
[발주기관] 한국가스공사
[요약] - 사업목적: 기술지원 종료에 대비한 ERP 업그레이드로 안정적 서비스 제공 및 업무생산성 향상
- 주요 사업내용: SAP ERP 전환, 업무 프로세스 개선, 생산공급 시스템 재구축, 전사 포털 및 업무 포털 신규 구축, 업무 프로세스 자산화 및 관리시스템 도입
- 신...
  [3] score=0.4874 | 경영정보시스템 기능개선 | (서두) > 2. 요구사항 세부내용
       [사업명] 경영정보시스템 기능개선
[발주기관] 부산관광공사
[요약] - 사업개요: 부산관광공사 경영정보시스템 기능개선 용역
- 추진배경: 업무환경 변화에 따른 기능개선 필요
- 사업범위: 경영정보시스템 기능 개선 및 최적화
- 기대효과: 업무처리 효율성 및 편의성 향상
- 추진목표: 사용자 요구사항 반영 및 업무환경 개선

[섹션: (서두) > 2. 요구사...

[질의] 사업 예산이 가장 큰 사업은?
  [1] score=0.2731 | 인천공항운영서비스㈜ 차세대 ERP시스템 구축 사업(재공고) | Ⅵ. 입찰 안내 사항 > 3. 상세 요구사항
       [사업명] 인천공항운영서비스㈜ 차세대 ERP시

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


[모델] ko-sroberta-multitask

[질의] 보안 요구사항이 있는 사업은?
  [1] score=0.0668 | 인천공항운영서비스㈜ 차세대 ERP시스템 구축 사업(재공고) | Ⅵ. 입찰 안내 사항 > 3. 상세 요구사항
       [사업명] 인천공항운영서비스㈜ 차세대 ERP시스템 구축 사업(재공고)
[발주기관] 인천공항운영서비스(주)
[요약] - 사업명: 인천공항운영서비스 차세대 ERP시스템 구축 사업
- 사업기간: 9개월 (안정화 기간 포함)
- 사업예산: 996,356,000원
- 계약방법: 제한경쟁입찰(협상에 의한 계약)
- 주요 시스템 구축-운영 현황: ERP 시스템과 그룹웨...
  [2] score=0.0668 | 인천공항운영서비스㈜ 차세대 ERP시스템 구축 사업(재공고) | Ⅵ. 입찰 안내 사항 > 3. 상세 요구사항
       [사업명] 인천공항운영서비스㈜ 차세대 ERP시스템 구축 사업(재공고)
[발주기관] 인천공항운영서비스(주)
[요약] - 사업명: 인천공항운영서비스 차세대 ERP시스템 구축 사업
- 사업기간: 9개월 (안정화 기간 포함)
- 사업예산: 996,356,000원
- 계약방법: 제한경쟁입찰(협상에 의한 계약)
- 주요 시스템 구축-운영 현황: ERP 시스템과 그룹웨...
  [3] score=0.0668 | 인천공항운영서비스㈜ 차세대 ERP시스템 구축 사업(재공고) | Ⅵ. 입찰 안내 사항 > 2. 입찰 및 계약 방식
       [사업명] 인천공항운영서비스㈜ 차세대 ERP시스템 구축 사업(재공고)
[발주기관] 인천공항운영서비스(주)
[요약] - 사업명: 인천공항운영서비스 차세대 ERP시스템 구축 사업
- 사업기간: 9개월 (안정화 기간 포함)
- 사업예산: 996,356,000원
- 계약방법: 제한경쟁입찰(협상에 의한 계약)
- 주요 시스템 구축-운영 현황: ERP 시스템과 그룹웨...

[질의] 사업 예산이 가장 큰 사업은?
  [1] score=0.0294 | 수도국산달동네박물관 전시해설 시스템 구축(협상에

7. 평가 (Recall@k / MRR)

In [29]:
golden_set = []
with open(GOLDEN_SET_PATH, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    golden_set = list(reader)

golden_set = golden_set[:101]
print(f"평가 문항 수: {len(golden_set)}")

평가 문항 수: 101


In [50]:
SCORE_THRESHOLD = None
import warnings
warnings.filterwarnings("ignore", message="Relevance scores must be between")


def hit(retrieved, k, target_ids):
    return any(tid in retrieved[:k] for tid in target_ids)


eval_results = []

for model in MODELS_TO_RUN:
    print(f"\n[시작] {model}")  # 루프 진입 확인
    safe_name = model.replace("/", "_")
    chroma_dir = f"{CHROMA_BASE}__{safe_name}"
    print(f"  경로: {chroma_dir}")
    print(f"  존재: {Path(chroma_dir).exists()}")

    if not Path(chroma_dir).exists():
        print(f"[SKIP] {model} — 저장된 DB 없음")
        continue

    print(f"\n[EVAL] {model}")
    vs = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=load_embedding_model(model),
        persist_directory=chroma_dir,
    )

    recall3, recall5, recall10 = [], [], []
    mrr_scores, weighted_mrr_scores = [], []
    query_times = []
    answer_scores = []
    skipped = 0

    for row in golden_set:
        gs_key = str(row["doc_id"])
        target = GS_TO_DOCID.get(gs_key)

        if target is None:
            skipped += 1
            continue

        target_ids = target if isinstance(target, list) else [target]
        question = row["question"]

        t0 = time.time()
        hits = vs.similarity_search_with_relevance_scores(question, k=10)
        query_times.append(time.time() - t0)

        # ── 점수 0~1 정규화 ──────────────────────────────────────────
        scores = [s for _, s in hits]
        min_s, max_s = min(scores), max(scores)
        if max_s > min_s:
            hits = [(doc, (s - min_s) / (max_s - min_s)) for doc, s in hits]
        else:
            hits = [(doc, 0.0) for doc, s in hits]

        # ── score 임계값 필터 ────────────────────────────────────────
        if SCORE_THRESHOLD is not None:
            hits = [(doc, score) for doc, score in hits if score >= SCORE_THRESHOLD]

        retrieved = [(doc.metadata.get("doc_id", ""), score) for doc, score in hits]
        retrieved_doc_ids = [doc_id for doc_id, _ in retrieved]

        # ── Recall ───────────────────────────────────────────────────
        recall3.append(1.0 if hit(retrieved_doc_ids, 3, target_ids) else 0.0)
        recall5.append(1.0 if hit(retrieved_doc_ids, 5, target_ids) else 0.0)
        recall10.append(1.0 if hit(retrieved_doc_ids, 10, target_ids) else 0.0)

        # ── MRR ──────────────────────────────────────────────────────
        rank = next(
            (i + 1 for i, d in enumerate(retrieved_doc_ids) if d in target_ids),
            None,
        )
        mrr_scores.append(1.0 / rank if rank else 0.0)

        # ── score 가중 MRR ───────────────────────────────────────────
        if rank:
            answer_score = retrieved[rank - 1][1]
            weighted_mrr_scores.append((1.0 / rank) * answer_score)
        else:
            weighted_mrr_scores.append(0.0)

        # ── avg_score (정답 청크 유사도) ─────────────────────────────
        matched_scores = [score for doc_id, score in retrieved if doc_id in target_ids]
        answer_scores.append(max(matched_scores) if matched_scores else 0.0)

    n = len(recall3)
    print(f"  평가: {n}개 | 제외: {skipped}개")

    eval_results.append({
        "model":         model,
        "Recall@3":      round(sum(recall3)              / n, 4),
        "Recall@5":      round(sum(recall5)              / n, 4),
        "Recall@10":     round(sum(recall10)             / n, 4),
        "MRR":           round(sum(mrr_scores)           / n, 4),
        "Weighted_MRR":  round(sum(weighted_mrr_scores)  / n, 4),
        "avg_ans_score": round(sum(answer_scores)        / n, 4),
        "avg_query_ms":  round(sum(query_times)          / n * 1000, 1),
        "비용/1M":       f"${MODEL_COST.get(model, 0)}",
    })


[시작] bge-m3
  경로: ../../data/vector_db__bge-m3
  존재: True

[EVAL] bge-m3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  평가: 84개 | 제외: 17개

[시작] ko-sroberta-multitask
  경로: ../../data/vector_db__ko-sroberta-multitask
  존재: True

[EVAL] ko-sroberta-multitask


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  평가: 84개 | 제외: 17개

[시작] text-embedding-3-small
  경로: ../../data/vector_db__text-embedding-3-small
  존재: True

[EVAL] text-embedding-3-small
  평가: 84개 | 제외: 17개


| model | Recall@3 | Recall@5 | Recall@10 | MRR | Weighted_MRR | avg_ans_score | avg_query_ms | 비용/1M |
|---|---|---|---|---|---|---|---|---|
| bge-m3 | 0.1667 | 0.1667 | 0.1786 | 0.1684 | 0.1674 | 0.1715 | 24.5 | 0.0 |
| ko-sroberta-multitask | 0.0714 | 0.0714 | 0.0714 | 0.0714 | 0.0119 | 0.0119 | 14.9 | 0.0 |
| text-embedding-3-small | 0.1786 | 0.2024 | 0.2381 | 0.1681 | 0.1542 | 0.1720 | 305.7 | 0.02 |

text-embedding-3-small > bge-m3 > ko-sroberta-multitask

bge-m3
속도: 24.5ms로 12배 빠름 → 실시간 응답에 유리
MRR 우위: 0.1684로 정답을 더 앞순위에 배치
무료: API 비용 없음
보안: 로컬 실행으로 데이터 외부 유출 없음


text-embedding-3-small
Recall@10 우위: 0.2381로 더 많은 정답 커버
k 확장 효과: k가 커질수록 성능 향상 (Recall@3→@10: +0.06)
간편한 배포: 로컬 GPU 불필요
유지보수: 모델 관리 불필요, API 호출만으로 사용